In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — CUSTOMERS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.customers AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id 
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_customers
)
SELECT
    customer_id,
    customer_unique_id,
    CAST(customer_zip_code_prefix AS STRING)    AS zip_code,
    UPPER(TRIM(customer_city))                  AS customer_city,
    UPPER(TRIM(customer_state))                 AS customer_state,
    current_timestamp()                         AS _processed_timestamp,
    'olist_customers'                           AS _source_table
FROM deduped
WHERE rn = 1
  AND customer_id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — SELLERS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.sellers AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY seller_id 
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_sellers
)
SELECT
    seller_id,
    CAST(seller_zip_code_prefix AS STRING)      AS zip_code,
    UPPER(TRIM(seller_city))                    AS seller_city,
    UPPER(TRIM(seller_state))                   AS seller_state,
    current_timestamp()                         AS _processed_timestamp,
    'olist_sellers'                             AS _source_table
FROM deduped
WHERE rn = 1
  AND seller_id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — PRODUCTS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.products AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY product_id 
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_products
),
translated AS (
    SELECT
        d.*,
        COALESCE(t.product_category_name_english, 'unknown') AS category_english
    FROM deduped d
    LEFT JOIN paymentdw.bronze.product_category_translation t
        ON d.product_category_name = t.product_category_name
    WHERE d.rn = 1
)
SELECT
    product_id,
    COALESCE(product_category_name, 'unknown')  AS category_portuguese,
    category_english,
    CAST(product_weight_g AS DECIMAL(10,2))     AS weight_g,
    CAST(product_length_cm AS DECIMAL(10,2))    AS length_cm,
    CAST(product_height_cm AS DECIMAL(10,2))    AS height_cm,
    CAST(product_width_cm AS DECIMAL(10,2))     AS width_cm,
    CAST(product_photos_qty AS INT)             AS photos_qty,
    CAST(product_name_lenght AS INT)            AS name_length,
    CAST(product_description_lenght AS INT)     AS description_length,
    current_timestamp()                         AS _processed_timestamp,
    'olist_products'                            AS _source_table
FROM translated
WHERE product_id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — ORDERS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.orders AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id 
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_orders
)
SELECT
    order_id,
    customer_id,
    TRIM(order_status)                                          AS order_status,
    TO_TIMESTAMP(order_purchase_timestamp)                      AS purchase_timestamp,
    TO_TIMESTAMP(order_approved_at)                             AS approved_at,
    TO_TIMESTAMP(order_delivered_carrier_date)                  AS delivered_carrier_date,
    TO_TIMESTAMP(order_delivered_customer_date)                 AS delivered_customer_date,
    TO_TIMESTAMP(order_estimated_delivery_date)                 AS estimated_delivery_date,
    -- business rule: was it delivered on time?
    CASE
        WHEN order_delivered_customer_date IS NOT NULL
         AND order_estimated_delivery_date IS NOT NULL
         AND TO_TIMESTAMP(order_delivered_customer_date) 
             <= TO_TIMESTAMP(order_estimated_delivery_date)
        THEN true
        ELSE false
    END                                                         AS is_delivered_on_time,
    -- business rule: how many days did delivery take?
    CASE
        WHEN order_delivered_customer_date IS NOT NULL
         AND order_purchase_timestamp IS NOT NULL
        THEN DATEDIFF(
            TO_TIMESTAMP(order_delivered_customer_date),
            TO_TIMESTAMP(order_purchase_timestamp))
        ELSE NULL
    END                                                         AS delivery_days,
    current_timestamp()                                         AS _processed_timestamp,
    'olist_orders'                                              AS _source_table
FROM deduped
WHERE rn = 1
  AND order_id IS NOT NULL
  AND customer_id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — ORDER ITEMS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.order_items AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id, order_item_id
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_order_items
)
SELECT
    order_id,
    CAST(order_item_id AS INT)                  AS order_item_id,
    product_id,
    seller_id,
    TO_TIMESTAMP(shipping_limit_date)           AS shipping_limit_date,
    CAST(price AS DECIMAL(12,2))                AS price,
    CAST(freight_value AS DECIMAL(12,2))        AS freight_value,
    CAST(price AS DECIMAL(12,2)) + 
        CAST(freight_value AS DECIMAL(12,2))    AS total_item_value,
    current_timestamp()                         AS _processed_timestamp,
    'olist_order_items'                         AS _source_table
FROM deduped
WHERE rn = 1
  AND order_id IS NOT NULL
  AND price > 0;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — ORDER PAYMENTS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.order_payments AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id, payment_sequential
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_order_payments
)
SELECT
    order_id,
    CAST(payment_sequential AS INT)             AS payment_sequential,
    TRIM(payment_type)                          AS payment_type,
    CAST(payment_installments AS INT)           AS payment_installments,
    CASE
        WHEN payment_value < 0  THEN NULL
        WHEN payment_value = 0  THEN NULL
        ELSE CAST(payment_value AS DECIMAL(12,2))
    END                                         AS payment_value,
    CASE
        WHEN payment_value < 0  THEN true
        ELSE false
    END                                         AS is_refund,
    current_timestamp()                         AS _processed_timestamp,
    'olist_order_payments'                      AS _source_table
FROM deduped
WHERE rn = 1
  AND order_id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — REVIEWS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.reviews AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY review_id
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_reviews
),
cleaned AS (
    SELECT *,
        try_cast(
            regexp_replace(CAST(review_score AS STRING), '[^0-9]', '')
        AS INT) AS review_score_clean
    FROM deduped
    WHERE rn = 1
)
SELECT
    review_id,
    order_id,
    CASE
        WHEN review_score_clean BETWEEN 1 AND 5 THEN review_score_clean
        ELSE NULL
    END                                         AS review_score,
    NULLIF(TRIM(review_comment_title), '')      AS review_title,
    NULLIF(TRIM(review_comment_message), '')    AS review_message,
    -- try_cast instead of to_timestamp
    -- malformed rows (text leaked in) become NULL instead of erroring
    try_cast(review_creation_date AS TIMESTAMP) AS review_created_at,
    try_cast(review_answer_timestamp AS TIMESTAMP) AS review_answered_at,
    current_timestamp()                         AS _processed_timestamp,
    'olist_reviews'                             AS _source_table
FROM cleaned
WHERE review_id IS NOT NULL
  AND order_id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — GEOLOCATION
-- (deduplicate by zip — keep one lat/lng per zip code)
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.geolocation AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY geolocation_zip_code_prefix
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.olist_geolocation
)
SELECT
    CAST(geolocation_zip_code_prefix AS STRING) AS zip_code,
    CAST(geolocation_lat AS DECIMAL(10,6))      AS latitude,
    CAST(geolocation_lng AS DECIMAL(10,6))      AS longitude,
    UPPER(TRIM(geolocation_city))               AS city,
    UPPER(TRIM(geolocation_state))              AS state,
    current_timestamp()                         AS _processed_timestamp,
    'olist_geolocation'                         AS _source_table
FROM deduped
WHERE rn = 1
  AND geolocation_zip_code_prefix IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — USERS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.users AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.users_data
)
SELECT
    CAST(id AS BIGINT)                          AS user_id,
    CAST(current_age AS INT)                    AS current_age,
    CAST(retirement_age AS INT)                 AS retirement_age,
    UPPER(TRIM(birth_year))                     AS birth_year,
    UPPER(TRIM(gender))                         AS gender,
    UPPER(TRIM(address))                        AS address,
    UPPER(TRIM(latitude))                       AS latitude,
    UPPER(TRIM(longitude))                      AS longitude,
    CAST(per_capita_income AS STRING)           AS per_capita_income_raw,
    try_cast(regexp_replace(
        per_capita_income, '[$,]', '') 
        AS DECIMAL(12,2))                       AS per_capita_income,
    try_cast(regexp_replace(
        yearly_income, '[$,]', '') 
        AS DECIMAL(12,2))                       AS yearly_income,
    try_cast(regexp_replace(
        total_debt, '[$,]', '') 
        AS DECIMAL(12,2))                       AS total_debt,
    CAST(credit_score AS INT)                   AS credit_score,
    CAST(num_credit_cards AS INT)               AS num_credit_cards,
    current_timestamp()                         AS _processed_timestamp,
    'users_data'                                AS _source_table
FROM deduped
WHERE rn = 1
  AND id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — CARDS
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.cards AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.cards_data
)
SELECT
    CAST(id AS BIGINT)                          AS card_id,
    CAST(client_id AS BIGINT)                   AS client_id,
    UPPER(TRIM(card_brand))                     AS card_brand,
    UPPER(TRIM(card_type))                      AS card_type,
    CAST(card_number AS STRING)                 AS card_number,
    CAST(expires AS STRING)                     AS expires,
    UPPER(TRIM(has_chip))                       AS has_chip,
    CAST(num_cards_issued AS INT)               AS num_cards_issued,
    try_cast(regexp_replace(
        credit_limit, '[$,]', '') 
        AS DECIMAL(12,2))                       AS credit_limit,
    CASE
        WHEN try_cast(regexp_replace(
            credit_limit, '[$,]', '') 
            AS DECIMAL(12,2)) = 0
        THEN true
        ELSE false
    END                                         AS is_limit_zero,
    CAST(acct_open_date AS STRING)              AS acct_open_date,
    CAST(year_pin_last_changed AS INT)          AS year_pin_last_changed,
    UPPER(TRIM(card_on_dark_web))               AS card_on_dark_web,
    current_timestamp()                         AS _processed_timestamp,
    'cards_data'                                AS _source_table
FROM deduped
WHERE rn = 1
  AND id IS NOT NULL;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — TRANSACTIONS
-- (largest table — 13.3M rows)
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.silver.transactions AS
WITH deduped AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM paymentdw.bronze.transactions_data
)
SELECT
    CAST(id AS BIGINT)                              AS transaction_id,
    CAST(client_id AS BIGINT)                       AS client_id,
    CAST(card_id AS BIGINT)                         AS card_id,
    try_cast(regexp_replace(
        amount, '[$,]', '') 
        AS DECIMAL(12,2))                           AS amount,
    CASE
        WHEN try_cast(regexp_replace(
            amount, '[$,]', '') 
            AS DECIMAL(12,2)) < 0
        THEN true ELSE false
    END                                             AS is_debit,
    TO_TIMESTAMP(date)                              AS transaction_date,
    CAST(merchant_id AS BIGINT)                     AS merchant_id,
    TRIM(merchant_city)                             AS merchant_city,
    UPPER(TRIM(merchant_state))                     AS merchant_state,
    CAST(zip AS STRING)                             AS zip,
    CAST(mcc AS INT)                                AS mcc_code,
    UPPER(TRIM(use_chip))                           AS use_chip,
    COALESCE(NULLIF(TRIM(errors), ''), 'No Error')  AS error_type,
    CASE
        WHEN errors IS NULL
          OR TRIM(errors) = ''   THEN false
        ELSE true
    END                                             AS has_error,
    current_timestamp()                             AS _processed_timestamp,
    'transactions_data'                             AS _source_table
FROM deduped
WHERE rn = 1
  AND id IS NOT NULL;

In [0]:
SELECT 'customers'      AS table_name, COUNT(*) AS row_count FROM paymentdw.silver.customers
UNION ALL SELECT 'sellers',            COUNT(*) FROM paymentdw.silver.sellers
UNION ALL SELECT 'products',           COUNT(*) FROM paymentdw.silver.products
UNION ALL SELECT 'orders',             COUNT(*) FROM paymentdw.silver.orders
UNION ALL SELECT 'order_items',        COUNT(*) FROM paymentdw.silver.order_items
UNION ALL SELECT 'order_payments',     COUNT(*) FROM paymentdw.silver.order_payments
UNION ALL SELECT 'reviews',            COUNT(*) FROM paymentdw.silver.reviews
UNION ALL SELECT 'geolocation',        COUNT(*) FROM paymentdw.silver.geolocation
UNION ALL SELECT 'users',              COUNT(*) FROM paymentdw.silver.users
UNION ALL SELECT 'cards',              COUNT(*) FROM paymentdw.silver.cards
UNION ALL SELECT 'transactions',       COUNT(*) FROM paymentdw.silver.transactions
ORDER BY table_name;